<a href="https://colab.research.google.com/github/tomo-mar/project_research_A/blob/main/Highlight_FewShot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

モデルのロード & Driveマウント

In [2]:
# 1. Driveマウントとディレクトリ設定
from google.colab import drive
import os

drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/VAD_Experiment"
INPUT_DIR = os.path.join(BASE_DIR, "input")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. ライブラリとモデルの準備
!pip install -q pydub librosa pandas numpy
!pip install -q git+https://github.com/huggingface/transformers@v4.51.3-Qwen2.5-Omni-preview
!pip install -q qwen-omni-utils

import torch
import librosa
import re
import pandas as pd
from pydub import AudioSegment
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor

import logging
logging.getLogger().setLevel(logging.ERROR)

model_path = "Qwen/Qwen2.5-Omni-7B"
model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    model_path, torch_dtype=torch.bfloat16, device_map="auto", attn_implementation="sdpa",
)
processor = Qwen2_5OmniProcessor.from_pretrained(model_path)
print("\n初期設定とモデルロード完了")

Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 79.6 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able 

config.json: 0.00B [00:00, ?B/s]

Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]


初期設定とモデルロード完了


few-shotデータ抽出関数の定義

In [3]:
def prepare_few_shot_examples(input_dir, train_ids=["rec01", "rec02", "rec03"]):
    """学習データから代表的なFew-shotサンプルを自動抽出する"""
    print("Few-shotサンプルを抽出中")
    examples = []

    # 高スコア(例: 0.8以上)と低スコア(例: 0.2以下)を1つずつ抽出する方針
    pos_found, neg_found = False, False

    for rec_id in train_ids:
        csv_path = os.path.join(input_dir, f"{rec_id}_label.csv")
        wav_path = os.path.join(input_dir, f"{rec_id}.wav")

        if not (os.path.exists(csv_path) and os.path.exists(wav_path)):
            continue

        df = pd.read_csv(csv_path)
        audio = AudioSegment.from_wav(wav_path)

        # ハイライトの抽出 (スコアが最も高いもの)
        if not pos_found:
            pos_row = df[df['Label_norm'] >= 0.8].head(1)
            if not pos_row.empty:
                time_sec = float(pos_row['Relative_Time_sec'].iloc[0])
                score = float(pos_row['Label_norm'].iloc[0])
                start_ms = int(time_sec * 1000)
                chunk = audio[start_ms:start_ms+5000]
                save_path = f"/content/shot_pos.wav"
                chunk.export(save_path, format="wav")
                examples.append({"path": save_path, "score": score})
                pos_found = True
                print(f"  - 抽出 [高スコア]: {rec_id} ({time_sec}秒, スコア:{score})")

        # 非ハイライトの抽出 (スコアが最も低いもの)
        if not neg_found:
            neg_row = df[df['Label_norm'] <= 0.2].head(1)
            if not neg_row.empty:
                time_sec = float(neg_row['Relative_Time_sec'].iloc[0])
                score = float(neg_row['Label_norm'].iloc[0])
                start_ms = int(time_sec * 1000)
                chunk = audio[start_ms:start_ms+5000]
                save_path = f"/content/shot_neg.wav"
                chunk.export(save_path, format="wav")
                examples.append({"path": save_path, "score": score})
                neg_found = True
                print(f"  - 抽出 [低スコア]: {rec_id} ({time_sec}秒, スコア:{score})")

        if pos_found and neg_found:
            break

    return examples

プロンプトと推論関数の定義

In [4]:
# プロンプトの定義
PROMPT_DIRECT = """
あなたは優秀な動画解析アシスタントです。
入力された5秒間の音声クリップを聞き、配信のハイライト（盛り上がり）確率を判定してください。

【評価基準】
0.0に近いほどベースライン（通常のトーン、特筆すべき起伏なし）
1.0に近いほどピーク（絶叫、爆笑など、最高潮の瞬間）

【出力形式】
判定結果となる「0.0 から 1.0 の間の数値（小数点以下2桁）」のみを出力してください。
"""

def run_qwen_few_shot(target_filepath, few_shot_examples):
    """Few-shot例を含めたマルチターン推論を実行"""
    messages = [
        {"role": "system", "content": [{"type": "text", "text": "You are a highly capable audio analysis AI."}]}
    ]
    audio_arrays = []

    # 1. Few-shotをプロンプトに履歴として追加
    for shot in few_shot_examples:
        messages.append({
            "role": "user", "content": [
                {"type": "audio", "audio_url": shot["path"]},
                {"type": "text", "text": PROMPT_DIRECT}
            ]
        })
        messages.append({
            "role": "assistant", "content": [
                {"type": "text", "text": str(shot["score"])}
            ]
        })
        arr, _ = librosa.load(shot["path"], sr=16000)
        audio_arrays.append(arr)

    # 2. 推論したいターゲット音声を追加
    messages.append({
        "role": "user", "content": [
            {"type": "audio", "audio_url": target_filepath},
            {"type": "text", "text": PROMPT_DIRECT}
        ]
    })
    target_arr, _ = librosa.load(target_filepath, sr=16000)
    audio_arrays.append(target_arr)

    # 推論処理
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=text, audio=audio_arrays, return_tensors="pt", padding=True).to("cuda")

    generate_ids = model.generate(**inputs, max_new_tokens=20, return_audio=False)
    generate_ids = [cur_ids[len(inputs.input_ids[0]):] for cur_ids in generate_ids]
    output_text = processor.batch_decode(generate_ids, skip_special_tokens=True)[0]

    return output_text

def extract_probability(text):
    matches = re.findall(r"0\.\d+|1\.0+", text)
    if matches:
        return float(matches[-1])
    return 0.0

メイン処理

In [5]:
# 1. 学習データからFew-shot用の音声を準備
few_shot_examples = prepare_few_shot_examples(INPUT_DIR, train_ids=["rec01", "rec02", "rec03"])

# 2. ターゲット音声の推論ループ
target_id = "rec04"
target_wav = os.path.join(INPUT_DIR, f"{target_id}.wav")

print(f"\n{target_id} のFew-shot推論を開始")
audio = AudioSegment.from_wav(target_wav)
results = []
chunk_length_ms = 5000

total_chunks = sum(1 for s in range(0, len(audio), chunk_length_ms) if len(audio[s:min(s + chunk_length_ms, len(audio))]) >= 1000)
current_chunk = 0

for start_ms in range(0, len(audio), chunk_length_ms):
    end_ms = min(start_ms + chunk_length_ms, len(audio))
    chunk = audio[start_ms:end_ms]

    if len(chunk) < 1000:
        continue

    current_chunk += 1
    time_sec = start_ms / 1000.0
    print(f"  [{current_chunk}/{total_chunks}] 解析中")

    temp_chunk_path = "/content/temp_chunk.wav"
    chunk.export(temp_chunk_path, format="wav")

    # Few-shot推論を実行
    res_text = run_qwen_few_shot(temp_chunk_path, few_shot_examples)
    score = extract_probability(res_text)

    results.append({
        "Time_sec": time_sec,
        "Score_Direct_FewShot": score,
        "RawText": res_text
    })

# CSV保存
df = pd.DataFrame(results)
output_csv_path = os.path.join(OUTPUT_DIR, f"{target_id}_fewshot.csv")
df.to_csv(output_csv_path, index=False)
print(f"\n結果を {output_csv_path} に保存完了")

Few-shotサンプルを抽出中
  - 抽出 [高スコア]: rec01 (80.0秒, スコア:0.875)
  - 抽出 [低スコア]: rec01 (0.0秒, スコア:0.0)

rec04 のFew-shot推論を開始
  [1/36] 解析中
  [2/36] 解析中
  [3/36] 解析中
  [4/36] 解析中
  [5/36] 解析中
  [6/36] 解析中
  [7/36] 解析中
  [8/36] 解析中
  [9/36] 解析中
  [10/36] 解析中
  [11/36] 解析中
  [12/36] 解析中
  [13/36] 解析中
  [14/36] 解析中
  [15/36] 解析中
  [16/36] 解析中
  [17/36] 解析中
  [18/36] 解析中
  [19/36] 解析中
  [20/36] 解析中
  [21/36] 解析中
  [22/36] 解析中
  [23/36] 解析中
  [24/36] 解析中
  [25/36] 解析中
  [26/36] 解析中
  [27/36] 解析中
  [28/36] 解析中
  [29/36] 解析中
  [30/36] 解析中
  [31/36] 解析中
  [32/36] 解析中
  [33/36] 解析中
  [34/36] 解析中
  [35/36] 解析中
  [36/36] 解析中

結果を /content/drive/MyDrive/VAD_Experiment/output/rec04_fewshot.csv に保存完了
